# 22.9 — Hierarchical & partial-order planning

Hierarchy chooses useful abstractions; partial-order planning delays sequence choices until constraints require them.

HTNs decompose abstract tasks into primitives, while partial-order planning stores only necessary before-after constraints. Together they reduce irrelevant branching and preserve concurrency.

Save a copy to Drive to edit.

In [ ]:
import itertools
import math
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple

import matplotlib.pyplot as plt


@dataclass
class Method:
    task: str
    subtasks: List[str]


def decompose_htn(task, methods, primitives):
    expanded = 0

    def expand(item):
        nonlocal expanded
        expanded += 1
        if item in primitives:
            return [item]
        choices = [method for method in methods if method.task == item]
        if not choices:
            raise ValueError(f"no method for {item}")
        result = []
        for subtask in choices[0].subtasks:
            result.extend(expand(subtask))
        return result

    plan = expand(task)
    return plan, expanded


def valid_orders(steps, constraints):
    orders = []
    for order in itertools.permutations(steps):
        position = {step: index for index, step in enumerate(order)}
        ok = True
        for before, after in constraints:
            if position[before] >= position[after]:
                ok = False
                break
        if ok:
            orders.append(order)
    return orders


def repair_threats(constraints, threats):
    repaired = set(constraints)
    for threat, producer, consumer in threats:
        if (producer, consumer) in repaired:
            repaired.add((consumer, threat))
    return repaired


def partial_order_plans(steps, constraints, threats=None):
    base = set(constraints)
    if threats:
        base = repair_threats(base, threats)
    orders = valid_orders(steps, base)
    return orders, base


def make_htn_problem(name, depth, width, bad_method=False, overconstrained=False):
    primitives = {f"Do{level}_{choice}" for level in range(depth) for choice in range(width)}
    methods = []
    previous = "Root"
    for level in range(depth):
        task = previous
        sub = f"Task{level}" if level < depth - 1 else f"Do{level}_0"
        methods.append(Method(task, [sub]))
        previous = sub
    if bad_method:
        methods.insert(0, Method("Root", ["MissingTask"]))
    steps = [f"S{i}" for i in range(min(depth + 2, 7))]
    constraints = {(steps[i], steps[i + 1]) for i in range(max(1, len(steps) - 2))}
    if overconstrained:
        constraints = set(itertools.permutations(steps, 2))
    threats = [(steps[-1], steps[0], steps[1])] if len(steps) > 2 else []
    return {"name": name, "methods": methods, "primitives": primitives, "steps": steps, "constraints": constraints, "threats": threats}


def pop_ladder():
    travel_methods = [Method("Travel", ["Pack", "Drive", "Park"])]
    travel_primitives = {"Pack", "Drive", "Park"}
    return [
        {"name": "D1 Travel POP", "methods": travel_methods, "primitives": travel_primitives, "steps": ["Buy", "Cook", "SetTable"], "constraints": {("Buy", "Cook")}, "threats": []},
        make_htn_problem("D2 wider methods", 3, 3),
        make_htn_problem("D3 nested hierarchy", 5, 3),
        make_htn_problem("D4 threats", 5, 4),
        make_htn_problem("D5 bad methods overconstraints", 6, 5, bad_method=False, overconstrained=False),
    ]

## The concept, built once (D1)

An HTN method maps an abstract task to subtasks. A partial-order plan has constraints $a\prec b$ and only commits to orders implied by causal structure.

In [ ]:
methods = [Method("Travel", ["Pack", "Drive", "Park"])]
primitives = {"Pack", "Drive", "Park"}
plan, expanded = decompose_htn("Travel", methods, primitives)
print(plan, expanded)
assert plan == ["Pack", "Drive", "Park"]
assert len(plan) == 3

Branching reduction: flat search has $6\cdot6\cdot6=216$ sequences; the HTN route has $2\cdot3=6$, a factor $36$. With Buy, Cook, SetTable and Buy $\prec$ Cook, $3$ of $3!=6$ orders are valid.

In [ ]:
flat = 6 * 6 * 6
htn = 2 * 3
reduction = flat // htn
orders, constraints = partial_order_plans(["Buy", "Cook", "SetTable"], {("Buy", "Cook")})
print("reduction", reduction, "valid orders", len(orders), orders)
assert flat == 216
assert htn == 6
assert reduction == 36
assert len(orders) == 3

## The dataset ladder

D1–D5 increase HTN depth, method width, threat repair, and ordering pressure while retaining a real partial-order schedule count.

In [ ]:
ladder = pop_ladder()
for problem in ladder:
    print(problem["name"], "methods", len(problem["methods"]), "steps", len(problem["steps"]), "constraints", len(problem["constraints"]), "threats", len(problem["threats"]))

## Run the SAME method across D1–D5

Metric: valid plan found, HTN expansion nodes, and ordering-flexibility count.

In [ ]:
results = []
for problem in ladder:
    root = problem["methods"][0].task
    try:
        plan, expanded = decompose_htn(root, problem["methods"], problem["primitives"])
        found = True
    except ValueError:
        plan = []
        expanded = 0
        found = False
    orders, constraints = partial_order_plans(problem["steps"], problem["constraints"], problem["threats"])
    results.append({"name": problem["name"], "found": found, "expanded": expanded, "flex": len(orders), "plan_len": len(plan)})
print("rung | found | expanded | primitive length | valid orders")
for row in results:
    print(row["name"], row["found"], row["expanded"], row["plan_len"], row["flex"])

## Results visualization

Panels show HTN expansion and remaining partial-order flexibility. The curve summarizes nodes and valid linearizations.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for axis, row in zip(axes, results):
    axis.bar(["expanded", "orders"], [row["expanded"], row["flex"]], color=["mediumpurple", "seagreen"])
    axis.set_title(row["name"].split()[0])
plt.show()
xs = list(range(1, len(results) + 1))
fig, axis = plt.subplots(figsize=(6, 3))
axis.plot(xs, [row["expanded"] for row in results], marker="o", label="HTN expanded")
axis.plot(xs, [row["flex"] for row in results], marker="s", label="valid orders")
axis.legend()
axis.set_xlabel("D rung")
plt.show()

## Pitfall on D5: overconstraining the plan

Unnecessary $a\prec b$ constraints destroy useful concurrency. The fix is to keep only causal and threat-repair constraints.

In [ ]:
steps = ladder[-1]["steps"]
minimal_orders, _ = partial_order_plans(steps, ladder[-1]["constraints"], ladder[-1]["threats"])
overconstraints = set(itertools.permutations(steps, 2))
over_orders, _ = partial_order_plans(steps, overconstraints)
print("minimal valid orders", len(minimal_orders))
print("overconstrained valid orders", len(over_orders))
assert len(minimal_orders) >= len(over_orders)

## Evaluate it + Practice

- Metric: valid plan found, nodes, and ordering flexibility.
- No-skill baseline: flat primitive enumeration.
- Cheap sanity check: Travel decomposes to 3 primitives; branching factor reduction is 36.
- Ablation: add unnecessary constraints and valid orders collapse.
- Failure signals: method library cannot decompose root or threats break causal links.

### Practice
Add a second Travel method and compare HTN expanded nodes.

### Practice
Introduce one threat and inspect repaired constraints.

### Practice
Count valid orders after adding SetTable before Cook.